# 07 Similarity Analysis

## Purpose

This notebook is intentionally simple:
- choose one saved `best_model/` checkpoint
- compare a few hand-picked glycan pairs
- build a similarity matrix for a small set of glycans
- display the heatmap inline and save the outputs to Drive

This is a manual inference notebook. It does not automatically pull examples from the train, validation, or test split.


## Setup note

- code stays in GitHub
- checkpoints and similarity outputs stay in Google Drive
- Colab pulls the repo at the start so the notebook uses the current GitHub version of `src/similarity.py`


In [ ]:
# ==============================================================================
# 0. SET UP THE COLAB ENVIRONMENT
# ==============================================================================
import os
import sys

from google.colab import drive

drive.mount('/content/drive')

GITHUB_OWNER = 'hb791-dev'
REPO_NAME = 'glycan-roberta'
REPO_URL = f'https://github.com/{GITHUB_OWNER}/{REPO_NAME}.git'
REPO_DIR = f'/content/{REPO_NAME}'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    print(f'Reusing existing repo at {REPO_DIR}')

%cd {REPO_DIR}
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)


In [ ]:
# ==============================================================================
# 1. IMPORT ANALYSIS HELPERS AND DEFINE DRIVE PATHS
# ==============================================================================
from pathlib import Path

from IPython.display import display

from src.similarity import (
    collect_preview_sequences,
    load_similarity_artifacts,
    run_similarity_analysis,
    validate_similarity_inputs,
)

DRIVE_ROOT = Path('/content/drive/MyDrive/ProjectRoot')
CHECKPOINTS_DIR = DRIVE_ROOT / 'checkpoints'
SIMILARITY_RESULTS_DIR = DRIVE_ROOT / 'results' / 'similarity'
SIMILARITY_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f'Drive root: {DRIVE_ROOT}')
print(f'Checkpoints root: {CHECKPOINTS_DIR}')
print(f'Similarity results root: {SIMILARITY_RESULTS_DIR}')


## Choose one model

Edit only `MODEL_DIR` in the next cell when you want to switch checkpoints.


In [ ]:
# ==============================================================================
# 2. CHOOSE ONE MODEL CHECKPOINT
# ==============================================================================
# Point to one saved best_model directory in Google Drive.
MODEL_DIR = DRIVE_ROOT / 'checkpoints' / 'manual' / 'mlm15_L6_H512_A8_lr00001_ep100_setv1_train_only_v2_cont_lr5e-05_ep20' / 'best_model'

# Keep the output folder name aligned with the checkpoint that produced it.
TOKENIZER_FAMILY = MODEL_DIR.parent.parent.name
EXPERIMENT_NAME = MODEL_DIR.parent.name
OUTPUT_NAME = f'{TOKENIZER_FAMILY}__{EXPERIMENT_NAME}'
OUTPUT_DIR = SIMILARITY_RESULTS_DIR / OUTPUT_NAME

print(f'Model directory: {MODEL_DIR}')
print(f'Output directory: {OUTPUT_DIR}')


## Choose example glycans

This is the only content cell you should usually edit after `MODEL_DIR`.
Add or remove curated example pairs, update the custom pair, and optionally extend the matrix sequence list.


In [ ]:
# ==============================================================================
# 3. CONFIGURE EXAMPLE PAIRS AND MATRIX SEQUENCES
# ==============================================================================
# These are manual examples for qualitative similarity checks.
# They are written in the compact corpus-style notation used to train the tokenizers.
EXAMPLE_GROUPS = {
    'Very similar': [
        ('Galb1-4GlcNAc', 'Galb1-4GlcNAc'),
        ('Galb1-4GlcNAc', 'Galb1-3GlcNAc'),
        ('NeuAca2-3Galb1-4GlcNAc', 'NeuAca2-6Galb1-4GlcNAc'),
    ],
    'Less similar': [
        ('Galb1-4GlcNAc', 'Fuca1-2Gal'),
        ('Galb1-4GlcNAc', 'Mana1-3(Mana1-6)Manb1-4GlcNAc'),
        ('NeuAca2-3Galb1-4GlcNAc', 'Mana1-3(Galb1-4GlcNAcb1-2Mana1-6)Manb1-4GlcNAc'),
    ],
}

# Set this to False if you do not want to include the custom pair section.
INCLUDE_CUSTOM_PAIR = True
CUSTOM_SEQ1 = 'Galb1-4GlcNAc'
CUSTOM_SEQ2 = 'NeuAca2-3Galb1-4GlcNAc'

# Add extra glycans here if you want the matrix to include sequences beyond
# those already used in the pairwise examples.
EXTRA_MATRIX_SEQUENCES = []

SEQUENCE_PAIRS = []
for group_name, pairs in EXAMPLE_GROUPS.items():
    for pair_number, (seq1, seq2) in enumerate(pairs, start=1):
        SEQUENCE_PAIRS.append(
            {
                'group_name': group_name,
                'pair_name': f'{group_name.lower().replace(" ", "_")}_{pair_number}',
                'seq1': seq1,
                'seq2': seq2,
            }
        )

if INCLUDE_CUSTOM_PAIR:
    SEQUENCE_PAIRS.append(
        {
            'group_name': 'Custom pair',
            'pair_name': 'custom_pair',
            'seq1': CUSTOM_SEQ1,
            'seq2': CUSTOM_SEQ2,
        }
    )

# Reuse every sequence that appears in the curated pairs, then append any
# optional extras for the matrix and heatmap view.
MATRIX_SEQUENCES = collect_preview_sequences(SEQUENCE_PAIRS, EXTRA_MATRIX_SEQUENCES)

MAX_LENGTH = None
BATCH_SIZE = 32

print(f'Pairwise examples configured: {len(SEQUENCE_PAIRS)}')
print(f'Matrix sequences configured: {len(MATRIX_SEQUENCES)}')


In [ ]:
# ==============================================================================
# 4. RUN THE ANALYSIS, DISPLAY THE TABLES, AND SAVE THE OUTPUTS
# ==============================================================================
validate_similarity_inputs(
    model_dir=MODEL_DIR,
    sequence_pairs=SEQUENCE_PAIRS,
    matrix_sequences=MATRIX_SEQUENCES,
    output_dir=OUTPUT_DIR,
)

tokenizer, model, device = load_similarity_artifacts(str(MODEL_DIR))

results = run_similarity_analysis(
    tokenizer=tokenizer,
    model=model,
    sequence_pairs=SEQUENCE_PAIRS,
    matrix_sequences=MATRIX_SEQUENCES,
    output_dir=OUTPUT_DIR,
    output_name=OUTPUT_NAME,
    device=device,
    max_length=MAX_LENGTH,
    batch_size=BATCH_SIZE,
    model_dir=MODEL_DIR,
)

pair_results_df = results['pair_results_df']

# Older copies of src/similarity.py may not return the optional
# group_name column. Fall back to one combined table so the notebook
# still works even before every environment is fully refreshed.
if 'group_name' in pair_results_df.columns:
    for group_name, group_df in pair_results_df.groupby('group_name', sort=False):
        print(f'=== {group_name} ===')
        display(group_df.drop(columns=['group_name']))
else:
    print('=== Pairwise similarities ===')
    display(pair_results_df)

print('=== Tokenization preview ===')
display(results['tokenization_preview_df'])

print('=== Similarity matrix ===')
display(results['similarity_df'])

print('Saved outputs:')
for label, path in results['saved_paths'].items():
    print(f'- {label}: {path}')
